In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, precision_recall_curve
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

df = pd.read_csv("/content/sample_data/loan_prediction.csv")
print(df.shape)
df.head()

(614, 13)


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


Preprocessing Steps

In [10]:
df = pd.read_csv("/content/sample_data/loan_prediction.csv")

if 'Loan_ID' in df.columns:
    df.drop('Loan_ID', axis=1, inplace=True)
    print("Dropped 'Loan_ID' column.")

numerical_cols_impute = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']
categorical_cols_impute = ['Gender', 'Married', 'Dependents', 'Self_Employed'] # Education and Property_Area have no missing values

# Handle missing values by imputation (instead of dropping rows)
print("Missing values before imputation:\n", df.isnull().sum())

for col in numerical_cols_impute:
    if col in df.columns and df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"  Filled missing values in {col} with median: {median_val}")

for col in categorical_cols_impute:
    if col in df.columns and df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f"  Filled missing values in {col} with mode: {mode_val}")

# Handle missing values in Loan_Status before mapping
if 'Loan_Status' in df.columns and df['Loan_Status'].isnull().sum() > 0:
    mode_loan_status = df['Loan_Status'].mode()[0]
    df['Loan_Status'].fillna(mode_loan_status, inplace=True)
    print(f"  Filled missing values in Loan_Status with mode: {mode_loan_status}")

print("Missing values after imputation:\n", df.isnull().sum())

df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})

X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

# Identify categorical and numerical columns for preprocessing pipeline
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
    ])

Dropped 'Loan_ID' column.
Missing values before imputation:
 Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64
  Filled missing values in LoanAmount with median: 128.0
  Filled missing values in Loan_Amount_Term with median: 360.0
  Filled missing values in Credit_History with median: 1.0
  Filled missing values in Gender with mode: Male
  Filled missing values in Married with mode: Yes
  Filled missing values in Dependents with mode: 0
  Filled missing values in Self_Employed with mode: No
Missing values after imputation:
 Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_Histo

Handle Class Imbalance with SMOTE + Model Comparison

In [11]:
# 3. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Training set class distribution:\n", y_train.value_counts())
# 4. Models with SMOTE
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = {}

for name, model in models.items():
    # Pipeline with SMOTE
    pipe = ImbPipeline([
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42, sampling_strategy=0.8)),  # 80% ratio
        ('classifier', model)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    print(f"\n{'='*60}")
    print(f"Model: {name}")
    print("="*60)
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

    results[name] = {
        'report': classification_report(y_test, y_pred, output_dict=True),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'model': pipe
    }

Training set class distribution:
 Loan_Status
1    316
0    144
Name: count, dtype: int64

Model: Logistic Regression
              precision    recall  f1-score   support

           0       0.77      0.71      0.74        48
           1       0.87      0.91      0.89       106

    accuracy                           0.84       154
   macro avg       0.82      0.81      0.81       154
weighted avg       0.84      0.84      0.84       154

ROC-AUC: 0.8626

Model: Random Forest
              precision    recall  f1-score   support

           0       0.79      0.65      0.71        48
           1       0.85      0.92      0.89       106

    accuracy                           0.84       154
   macro avg       0.82      0.79      0.80       154
weighted avg       0.83      0.84      0.83       154

ROC-AUC: 0.8293

Model: Gradient Boosting
              precision    recall  f1-score   support

           0       0.74      0.60      0.67        48
           1       0.83      0.91      